# Masking
Random and domain-informed masking strategies for spectral patches.

In [ ]:
import torch
import sys, os
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import config

WAVE_MIN = 3600.0
WAVE_MAX = 9800.0

LINES = {
    'CaHK':    3945.0,
    'break4k': 4000.0,
    'Hbeta':   4861.0,
    'OIII_1':  4959.0,
    'OIII_2':  5007.0,
    'Halpha':  6563.0,
}

In [ ]:
def random_mask(batch_size, n_patches, mask_ratio=config.MASK_RATIO):
    """Generate a random boolean mask with a fixed number of masked positions per sample.

    Args:
        batch_size (int): Number of samples in the batch.
        n_patches (int): Total number of spectral patches per sample.
        mask_ratio (float): Fraction of patches to mask.

    Returns:
        torch.Tensor: Boolean mask of shape (batch_size, n_patches); True = masked.
    """
    n_masked = int(mask_ratio * n_patches)
    mask = torch.zeros(batch_size, n_patches, dtype=torch.bool)
    for i in range(batch_size):
        indices = torch.randperm(n_patches)[:n_masked]
        mask[i, indices] = True
    return mask

In [ ]:
def domain_informed_mask(batch_size, redshifts, n_patches, mask_ratio=config.MASK_RATIO):
    """Generate a mask biased toward patches containing known spectral lines.

    Observed wavelength of each line is lambda_rest * (1 + redshift).
    Patches near those positions receive higher sampling weight.

    Args:
        batch_size (int): Number of samples in the batch.
        redshifts (torch.Tensor or list): Redshift values of shape (batch_size,).
        n_patches (int): Total number of spectral patches per sample.
        mask_ratio (float): Fraction of patches to mask.

    Returns:
        torch.Tensor: Boolean mask of shape (batch_size, n_patches); True = masked.
    """
    n_masked = int(mask_ratio * n_patches)
    mask = torch.zeros(batch_size, n_patches, dtype=torch.bool)

    for i in range(batch_size):
        z = float(redshifts[i])
        weights = torch.ones(n_patches)

        for lambda_rest in LINES.values():
            lambda_obs = lambda_rest * (1.0 + z)
            if WAVE_MIN <= lambda_obs <= WAVE_MAX:
                patch_idx = int((lambda_obs - WAVE_MIN) / (WAVE_MAX - WAVE_MIN) * n_patches)
                patch_idx = min(patch_idx, n_patches - 1)
                lo = max(0, patch_idx - 5)
                hi = min(n_patches, patch_idx + 6)
                weights[lo:hi] += 3.0

        indices = torch.multinomial(weights, n_masked, replacement=False)
        mask[i, indices] = True

    return mask

In [ ]:
def apply_mask(patch_tokens, mask, mask_embedding):
    """Replace masked patch positions with the learned mask embedding.

    Args:
        patch_tokens (torch.Tensor): Shape (B, N_PATCHES, D_MODEL).
        mask (torch.Tensor): Boolean mask of shape (B, N_PATCHES); True = masked.
        mask_embedding (torch.Tensor): Learned embedding of shape (1, 1, D_MODEL).

    Returns:
        torch.Tensor: Patch tokens with masked positions replaced, shape (B, N_PATCHES, D_MODEL).
    """
    mask_expanded = mask.unsqueeze(-1).expand_as(patch_tokens)
    return torch.where(mask_expanded, mask_embedding.expand_as(patch_tokens), patch_tokens)